# Coordinate Plots
> Standalone plotting functions for coordinate-based visualizations (UMAP, spatial, etc.)

In [ ]:
#| default_exp coord_plots

In [ ]:
#| export
from __future__ import annotations

from typing import List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.sparse import issparse
from scipy.stats import gaussian_kde

In [ ]:
#| export
def _dense_X(X) -> np.ndarray:
    """Convert sparse or dense matrix to dense numpy array."""
    return np.asarray(X.toarray() if issparse(X) else X, dtype=float)


def _normal_reference_bandwidth(x):
    """Scott's rule for bandwidth selection."""
    x = np.asarray(x, float)
    std = np.std(x, ddof=1)
    n = len(x)
    return 1.06 * std * (n ** (-1/5))


def wkde2d(x, y, w=None, h=None, adjust=1.0, n=100, lims=None):
    """
    Weighted 2D KDE.

    Parameters
    ----------
    x, y : 1D arrays
        Coordinates
    w : 1D array, optional
        Weights (same length as x, y). If None, uniform weights.
    h : float or tuple, optional
        Bandwidth(s). If None, use Scott's rule.
    adjust : float
        Bandwidth multiplier
    n : int
        Number of grid points per axis
    lims : list, optional
        [xmin, xmax, ymin, ymax]

    Returns
    -------
    dict
        Dictionary with keys 'x', 'y', 'z' (grid coordinates and density)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if w is None:
        w = np.ones_like(x)
    else:
        w = np.asarray(w, dtype=float)

    if len(x) != len(y) or len(x) != len(w):
        raise ValueError("x, y, w must all have same length")

    if lims is None:
        lims = [x.min(), x.max(), y.min(), y.max()]
    if len(lims) != 4:
        raise ValueError("lims must be [xmin, xmax, ymin, ymax]")

    if h is None:
        hx = _normal_reference_bandwidth(x)
        hy = _normal_reference_bandwidth(y)
        h = (hx, hy)
    else:
        if np.isscalar(h):
            h = (float(h), float(h))
        else:
            h = tuple(h)

    h = (h[0] * adjust, h[1] * adjust)

    gx = np.linspace(lims[0], lims[1], n)
    gy = np.linspace(lims[2], lims[3], n)

    ax = (gx[:, None] - x[None, :]) / h[0]  # (n, N)
    ay = (gy[:, None] - y[None, :]) / h[1]  # (n, N)

    fx = np.exp(-0.5 * ax**2) / np.sqrt(2.0 * np.pi)
    fy = np.exp(-0.5 * ay**2) / np.sqrt(2.0 * np.pi)

    w = w[None, :]  # (1, N)
    fxw = fx * w
    fyw = fy * w

    z = fxw.dot(fyw.T)  # (n, n) - rows=x-grid, cols=y-grid

    Z = z / (w.sum() * h[0] * h[1])
    return {"x": gx, "y": gy, "z": Z}


def get_dens(points, dens):
    """
    Map each 2D point to the approximate density in dens["z"].
    
    Parameters
    ----------
    points : np.ndarray
        Points array (n_cells, 2)
    dens : dict
        Dictionary with 'x', 'y', 'z' from wkde2d
        
    Returns
    -------
    np.ndarray
        Density values at each point
    """
    xgrid = dens["x"]
    ygrid = dens["y"]
    Z = dens["z"]  # Shape: (len(xgrid), len(ygrid)) = (n_x, n_y)

    pts = np.asarray(points, float)
    ix = np.searchsorted(xgrid, pts[:, 0]) - 1
    iy = np.searchsorted(ygrid, pts[:, 1]) - 1

    ix = np.clip(ix, 0, len(xgrid) - 1)
    iy = np.clip(iy, 0, len(ygrid) - 1)

    # Z is indexed as Z[x_index, y_index]
    return Z[ix, iy]


def calculate_density(
    adata,
    feature: str,
    basis: str = "umap",
    coord_key: Optional[str] = None,
    adjust: float = 1.0,
    map_to_cells: bool = True,
    n: int = 200,
    lims=None,
) -> np.ndarray:
    """
    Calculate weighted KDE-based spatial density for a feature.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object
    feature : str
        Feature name (e.g., transcript ID)
    basis : str
        Embedding basis (e.g., 'umap', 'spatial')
    coord_key : str, optional
        Override for coordinate key in obsm
    adjust : float
        Bandwidth adjustment for KDE
    map_to_cells : bool
        If True, return density values for each cell
    n : int
        Grid resolution for density calculation
    lims : tuple, optional
        Coordinate limits [xmin, xmax, ymin, ymax]
        
    Returns
    -------
    np.ndarray
        Density values (per cell if map_to_cells=True, else grid)
    """
    # Get coordinates
    if coord_key is not None:
        if coord_key not in adata.obsm:
            raise ValueError(f"AnnData has no .obsm['{coord_key}']")
        coords = adata.obsm[coord_key]
    else:
        coord_key = f"X_{basis}"
        if coord_key not in adata.obsm:
            raise ValueError(f"Coordinate key '{coord_key}' not found in adata.obsm")
        coords = adata.obsm[coord_key]
    
    coords = np.asarray(coords)
    if coords.shape[1] < 2:
        raise ValueError(f"Coords must have at least 2 dims, got {coords.shape}")
    coords2d = coords[:, :2]
    
    # Get feature values as weights
    if feature in adata.var_names:
        w = adata.obs_vector(feature)
    elif feature in adata.obs.columns:
        w = adata.obs[feature].values
    else:
        raise ValueError(f"Feature '{feature}' not found in var_names or obs")
    
    w = np.asarray(w, dtype=float)
    
    # Compute weighted 2D density
    dens = wkde2d(
        x=coords2d[:, 0],
        y=coords2d[:, 1],
        w=w,
        adjust=adjust,
        n=n,
        lims=lims,
    )
    
    if map_to_cells:
        # Map grid back to cell positions
        return get_dens(coords2d, dens)
    else:
        # Return the full grid
        return dens

In [ ]:
#| export
def _versionless(tid: str) -> str:
    """Remove version suffix from transcript ID (e.g., ENST00000123.4 -> ENST00000123)."""
    i = tid.rfind(".")
    return tid[:i] if i > 0 and tid[i+1:].isdigit() else tid


def get_coords_from_adata(
    adata,
    basis: str = "umap",
    coord_key: Optional[str] = None,
) -> np.ndarray:
    """Extract 2D coordinates from AnnData."""
    if coord_key is None:
        coord_key = f"X_{basis}"
    if coord_key not in adata.obsm:
        raise ValueError(f"Coordinate key '{coord_key}' not found in adata.obsm")
    coords = np.asarray(adata.obsm[coord_key])
    if coords.shape[1] < 2:
        raise ValueError(f"Coordinates must have at least 2 dimensions, got {coords.shape[1]}")
    return coords[:, :2]


def get_expression_from_adata(
    adata,
    features: List[str],
) -> np.ndarray:
    """Extract expression values for specified features from AnnData."""
    if isinstance(features, str):
        features = [features]
    missing = [f for f in features if f not in adata.var_names]
    if missing:
        raise ValueError(f"Features not found in adata.var_names: {missing}")
    X = adata[:, features].X
    if issparse(X):
        X = X.toarray()
    return np.asarray(X.T, float)


def plot_coords_standalone(
    coords: np.ndarray,
    values: np.ndarray,
    *,
    titles: Optional[List[str]] = None,
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 8.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_per_panel_scale: bool = False,
    axis_labels: Tuple[str, str] = ("x", "y"),
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
) -> plt.Figure:
    """Standalone coordinate plot with multiple panels.

    Parameters
    ----------
    coords : np.ndarray
        Coordinates array (n_cells, 2)
    values : np.ndarray
        Values array (n_features, n_cells) or (n_cells,)
    titles : list of str, optional
        Title for each panel
    cmaps : list of str, optional
        Colormap for each panel
    max_cols : int
        Maximum columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Global color scale limits (ignored if use_per_panel_scale=True)
    use_per_panel_scale : bool
        If True, each panel has its own colorbar scale based on its own data.
        If False (default), all panels share a global scale.
    axis_labels : tuple of str
        Labels for x and y axes
    show_colorbar : bool
        Whether to show colorbar
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)

    Returns
    -------
    plt.Figure
        The figure object
    """
    values = np.asarray(values, float)
    if values.ndim == 1:
        values = values.reshape(1, -1)

    n_features = values.shape[0]

    if titles is None:
        titles = [f"Feature {i+1}" for i in range(n_features)]
    if cmaps is None:
        cmaps = ["viridis"] * n_features

    # Compute global vmin/vmax if not using per-panel scale
    if not use_per_panel_scale:
        if vmin is None:
            vmin = float(np.nanpercentile(values, 1.0))
        if vmax is None:
            vmax = float(np.nanpercentile(values, 99.0))

    ncols = min(max_cols, n_features)
    nrows = int(np.ceil(n_features / ncols))
    if fig_height is None:
        fig_height = (fig_width / ncols) * nrows + 0.5

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width, fig_height))
    if n_features == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i in range(n_features):
        ax = axes[i]
        vals = values[i]

        # Compute per-panel scale if requested
        if use_per_panel_scale:
            panel_vmin = float(np.nanpercentile(vals, 1.0))
            panel_vmax = float(np.nanpercentile(vals, 99.0))
        else:
            panel_vmin = vmin
            panel_vmax = vmax

        sc = ax.scatter(
            coords[:, 0], coords[:, 1],
            c=vals, s=size, cmap=cmaps[i],
            alpha=alpha, vmin=panel_vmin, vmax=panel_vmax,
            linewidths=0, rasterized=True,
        )

        if show_colorbar:
            # Use inset_axes for colorbar positioning
            cax = ax.inset_axes([1.02, 0.1, 0.03, 0.8])  # [x0, y0, w, h] in axes coords
            cb = fig.colorbar(sc, cax=cax)
            cb.ax.tick_params(labelsize=7)

        ax.set_aspect('equal')
        ax.set_title(titles[i], fontsize=10)
        ax.set_xlabel(axis_labels[0] if i // ncols == nrows - 1 else "", fontsize=9)
        ax.set_ylabel(axis_labels[1] if i % ncols == 0 else "", fontsize=9)
        ax.tick_params(labelsize=8)

    for i in range(n_features, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    return fig

In [ ]:
#| export
def plot_coords_from_adata(
    adata,
    features: List[str],
    *,
    basis: str = "umap",
    coord_key: Optional[str] = None,
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_global_vmin_vmax: bool = True,
    axis_labels: Optional[Tuple[str, str]] = None,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
) -> Tuple[plt.Figure, np.ndarray, np.ndarray]:
    """
    Plot coordinates from AnnData for multiple features.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    features : list of str
        Feature names (transcript IDs, gene names, etc.)
    basis : str
        Embedding basis (e.g., 'umap', 'tsne')
    coord_key : str, optional
        Override for coordinate key in obsm
    cmaps : list of str, optional
        Colormap for each feature
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits (overrides use_global_vmin_vmax)
    use_global_vmin_vmax : bool
        If True, use global min/max across all features
    axis_labels : tuple of str, optional
        Labels for x and y axes (default: based on basis)
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)

    Returns
    -------
    fig : plt.Figure
        The figure object
    coords : np.ndarray
        Coordinates array (n_cells, 2)
    values : np.ndarray
        Expression matrix (n_features, n_cells)
    """
    # Get coordinates
    coords = get_coords_from_adata(adata, basis=basis, coord_key=coord_key)

    # Get expression values
    values = get_expression_from_adata(adata, features)

    # Default axis labels based on basis
    if axis_labels is None:
        if basis.lower() == "umap":
            axis_labels = ("UMAP1", "UMAP2")
        elif basis.lower() == "tsne":
            axis_labels = ("tSNE1", "tSNE2")
        elif basis.lower() in ("spatial", "space"):
            axis_labels = ("X", "Y")
        else:
            axis_labels = (f"{basis}1", f"{basis}2")

    # Strip version suffixes from titles
    titles = [_versionless(f) for f in features]

    # Plot
    fig = plot_coords_standalone(
        coords,
        values,
        titles=titles,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        axis_labels=axis_labels,
        show_colorbar=show_colorbar,
        fig_width=fig_width,
        fig_height=fig_height,
    )

    return fig, coords, values

In [ ]:
#| export
def _resolve_transcripts(
    adata,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    group_col: Optional[str] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
) -> List[str]:
    """
    Resolve which transcripts to plot.

    Either takes explicit transcript list OR derives top_n transcripts
    from gene_id using group-based ranking.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    transcripts : list of str, optional
        Explicit list of transcript IDs
    gene_id : str, optional
        Gene ID to get top transcripts from
    top_n : int
        Number of top transcripts to select (when using gene_id)
    group_col : str, optional
        Column in adata.obs for grouping (required when using gene_id)
    estimator : str
        Estimator for PSI calculation ('pseudobulk' or 'mean')
    dirichlet_alpha : float
        Dirichlet alpha for pseudobulk estimation
    epsilon : float
        Small value to avoid division by zero

    Returns
    -------
    List[str]
        List of transcript IDs to plot
    """
    if transcripts is not None:
        # Path 1: User provided explicit list
        return transcripts if isinstance(transcripts, list) else [transcripts]
    elif gene_id is not None:
        # Path 2: Get top_n transcripts for this gene
        if group_col is None:
            raise ValueError("group_col is required when using gene_id")

        from allos.quant_plots import _compute_group_matrix_from_adata

        iso_ids, groups, V = _compute_group_matrix_from_adata(
            adata, gene_id, group_col,
            top_n=top_n, estimator=estimator,
            dirichlet_alpha=dirichlet_alpha, epsilon=epsilon
        )
        return iso_ids
    else:
        raise ValueError("Must provide either 'transcripts' or 'gene_id'")

In [ ]:
#| export
def plot_transcript_umap(
    adata,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    group_col: Optional[str] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    basis: str = "umap",
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_global_vmin_vmax: bool = True,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
) -> plt.Figure:
    """
    Plot transcript expression on UMAP embedding (scanpy-like API).

    Provide either explicit transcripts OR gene_id + top_n to automatically
    select the most variable isoforms.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot
    gene_id : str, optional
        Gene ID to get top transcripts from
    top_n : int
        Number of top transcripts to select (when using gene_id)
    group_col : str, optional
        Column in adata.obs for grouping (required when using gene_id)
    estimator : str
        Estimator for PSI calculation ('pseudobulk' or 'mean')
    dirichlet_alpha : float
        Dirichlet alpha for pseudobulk estimation
    epsilon : float
        Small value to avoid division by zero
    basis : str
        Embedding basis (default: 'umap')
    cmaps : list of str, optional
        Colormap for each transcript (default: viridis variants)
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits
    use_global_vmin_vmax : bool
        If True, use global min/max across all transcripts
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)

    Returns
    -------
    plt.Figure
        The matplotlib figure

    Examples
    --------
    # Explicit transcripts
    fig = plot_transcript_umap(adata, transcripts=["ENST001.1", "ENST002.1"])

    # Auto-select top 3 isoforms for a gene
    fig = plot_transcript_umap(adata, gene_id="Myl6", top_n=3, group_col="cell_type")
    """
    # Resolve which transcripts to plot
    transcript_ids = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
        epsilon=epsilon,
    )

    if not transcript_ids:
        raise ValueError("No transcripts found")

    # Default colormaps: all viridis (matches composed plots)
    if cmaps is None:
        cmaps = ["viridis"] * len(transcript_ids)

    # Plot using existing function
    fig, coords, values = plot_coords_from_adata(
        adata,
        features=transcript_ids,
        basis=basis,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        use_global_vmin_vmax=use_global_vmin_vmax,
        show_colorbar=show_colorbar,
        fig_width=fig_width,
        fig_height=fig_height,
    )

    return fig

In [ ]:
#| export
def plot_umap_density(
    adata,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    group_col: Optional[str] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    basis: str = "umap",
    adjust: float = 1.0,
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_per_panel_scale: bool = True,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
) -> plt.Figure:
    """
    Plot transcript spatial density (KDE) on UMAP embedding (scanpy-like API).

    Provide either explicit transcripts OR gene_id + top_n to automatically
    select the most variable isoforms.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot
    gene_id : str, optional
        Gene ID to get top transcripts from
    top_n : int
        Number of top transcripts to select (when using gene_id)
    group_col : str, optional
        Column in adata.obs for grouping (required when using gene_id)
    estimator : str
        Estimator for PSI calculation ('pseudobulk' or 'mean')
    dirichlet_alpha : float
        Dirichlet alpha for pseudobulk estimation
    epsilon : float
        Small value to avoid division by zero
    basis : str
        Embedding basis (default: 'umap')
    adjust : float
        Bandwidth adjustment for KDE
    cmaps : list of str, optional
        Colormap for each transcript (default: Reds variants)
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits (only used if use_per_panel_scale=False)
    use_per_panel_scale : bool
        If True (default), each panel has its own colorbar scale.
        If False, all panels share a global scale.
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)

    Returns
    -------
    plt.Figure
        The matplotlib figure

    Examples
    --------
    # Explicit transcripts
    fig = plot_umap_density(adata, transcripts=["ENST001.1", "ENST002.1"])

    # Auto-select top 3 isoforms for a gene
    fig = plot_umap_density(adata, gene_id="Clta", top_n=3, group_col="cell_type")
    """
    # Resolve which transcripts to plot
    transcript_ids = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
        epsilon=epsilon,
    )

    if not transcript_ids:
        raise ValueError("No transcripts found")

    # Default colormaps: all Reds (matches composed plots)
    if cmaps is None:
        cmaps = ["Reds"] * len(transcript_ids)

    # Plot using existing function
    fig, coords, density_values = plot_density_from_adata(
        adata,
        features=transcript_ids,
        basis=basis,
        adjust=adjust,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        use_per_panel_scale=use_per_panel_scale,
        show_colorbar=show_colorbar,
        fig_width=fig_width,
        fig_height=fig_height,
    )

    return fig


def plot_transcript_spatial(
    adata,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    group_col: Optional[str] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    basis: str = "spatial",
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_global_vmin_vmax: bool = True,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
    invert_y: bool = True,
) -> plt.Figure:
    """
    Plot transcript expression on spatial coordinates (scanpy-like API).

    Provide either explicit transcripts OR gene_id + top_n to automatically
    select the most variable isoforms.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot
    gene_id : str, optional
        Gene ID to get top transcripts from
    top_n : int
        Number of top transcripts to select (when using gene_id)
    group_col : str, optional
        Column in adata.obs for grouping (required when using gene_id)
    estimator : str
        Estimator for PSI calculation ('pseudobulk' or 'mean')
    dirichlet_alpha : float
        Dirichlet alpha for pseudobulk estimation
    epsilon : float
        Small value to avoid division by zero
    basis : str
        Embedding basis (default: 'spatial')
    cmaps : list of str, optional
        Colormap for each transcript (default: viridis variants)
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits
    use_global_vmin_vmax : bool
        If True, use global min/max across all transcripts
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    invert_y : bool
        If True, invert y-axis (default: True for spatial plots)

    Returns
    -------
    plt.Figure
        The matplotlib figure

    Examples
    --------
    # Explicit transcripts
    fig = plot_transcript_spatial(adata, transcripts=["ENST001.1", "ENST002.1"])

    # Auto-select top 3 isoforms for a gene
    fig = plot_transcript_spatial(adata, gene_id="Myl6", top_n=3, group_col="cell_type")
    """
    # Resolve which transcripts to plot
    transcript_ids = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
        epsilon=epsilon,
    )

    if not transcript_ids:
        raise ValueError("No transcripts found")

    # Default colormaps: all viridis (matches composed plots)
    if cmaps is None:
        cmaps = ["viridis"] * len(transcript_ids)

    # Plot using existing function
    fig, coords, values = plot_coords_from_adata(
        adata,
        features=transcript_ids,
        basis=basis,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        use_global_vmin_vmax=use_global_vmin_vmax,
        show_colorbar=show_colorbar,
        fig_width=fig_width,
        fig_height=fig_height,
    )

    # Invert y-axis for spatial plots if requested
    if invert_y:
        for ax in fig.axes:
            ax.invert_yaxis()

    return fig


def plot_spatial_density(
    adata,
    *,
    transcripts: Optional[List[str]] = None,
    gene_id: Optional[str] = None,
    top_n: int = 2,
    group_col: Optional[str] = None,
    estimator: str = "pseudobulk",
    dirichlet_alpha: float = 0.5,
    epsilon: float = 1e-6,
    basis: str = "spatial",
    adjust: float = 1.0,
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_per_panel_scale: bool = True,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
    invert_y: bool = True,
) -> plt.Figure:
    """
    Plot transcript spatial density (KDE) on spatial coordinates (scanpy-like API).

    Provide either explicit transcripts OR gene_id + top_n to automatically
    select the most variable isoforms.

    Parameters
    ----------
    adata : AnnData
        Annotated data object
    transcripts : list of str, optional
        Explicit list of transcript IDs to plot
    gene_id : str, optional
        Gene ID to get top transcripts from
    top_n : int
        Number of top transcripts to select (when using gene_id)
    group_col : str, optional
        Column in adata.obs for grouping (required when using gene_id)
    estimator : str
        Estimator for PSI calculation ('pseudobulk' or 'mean')
    dirichlet_alpha : float
        Dirichlet alpha for pseudobulk estimation
    epsilon : float
        Small value to avoid division by zero
    basis : str
        Embedding basis (default: 'spatial')
    adjust : float
        Bandwidth adjustment for KDE
    cmaps : list of str, optional
        Colormap for each transcript (default: Reds variants)
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits (only used if use_per_panel_scale=False)
    use_per_panel_scale : bool
        If True (default), each panel has its own colorbar scale.
        If False, all panels share a global scale.
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    invert_y : bool
        If True, invert y-axis (default: True for spatial plots)

    Returns
    -------
    plt.Figure
        The matplotlib figure

    Examples
    --------
    # Explicit transcripts
    fig = plot_spatial_density(adata, transcripts=["ENST001.1", "ENST002.1"])

    # Auto-select top 3 isoforms for a gene
    fig = plot_spatial_density(adata, gene_id="Clta", top_n=3, group_col="cell_type")
    """
    # Resolve which transcripts to plot
    transcript_ids = _resolve_transcripts(
        adata,
        transcripts=transcripts,
        gene_id=gene_id,
        top_n=top_n,
        group_col=group_col,
        estimator=estimator,
        dirichlet_alpha=dirichlet_alpha,
        epsilon=epsilon,
    )

    if not transcript_ids:
        raise ValueError("No transcripts found")

    # Default colormaps: all Reds (matches composed plots)
    if cmaps is None:
        cmaps = ["Reds"] * len(transcript_ids)

    # Plot using existing function
    fig, coords, density_values = plot_density_from_adata(
        adata,
        features=transcript_ids,
        basis=basis,
        adjust=adjust,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        use_per_panel_scale=use_per_panel_scale,
        show_colorbar=show_colorbar,
        fig_width=fig_width,
        fig_height=fig_height,
    )

    # Invert y-axis for spatial plots if requested
    if invert_y:
        for ax in fig.axes:
            ax.invert_yaxis()

    return fig

In [ ]:
#| export
def plot_umap_from_adata(
    adata,
    features: List[str],
    *,
    basis: str = "umap",
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_global_vmin_vmax: bool = True,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
) -> Tuple[plt.Figure, np.ndarray, np.ndarray]:
    """
    Convenience wrapper: plot features on UMAP embedding.
    
    This is an alias for plot_coords_from_adata with sensible UMAP defaults.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object
    features : list of str
        Feature names (transcript IDs, gene names, etc.)
    basis : str
        Embedding basis (default: 'umap')
    cmaps : list of str, optional
        Colormap for each feature
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits
    use_global_vmin_vmax : bool
        If True, use global min/max across all features
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    
    Returns
    -------
    fig : plt.Figure
        The figure object
    coords : np.ndarray
        Coordinates array (n_cells, 2)
    values : np.ndarray
        Expression matrix (n_features, n_cells)
    """
    return plot_coords_from_adata(
        adata,
        features,
        basis=basis,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        use_global_vmin_vmax=use_global_vmin_vmax,
        show_colorbar=show_colorbar,
        axis_labels=None,  # Auto-detect from basis
        fig_width=fig_width,
        fig_height=fig_height,
    )


def plot_density_from_adata(
    adata,
    features: List[str],
    *,
    basis: str = "umap",
    adjust: float = 1.0,
    cmaps: Optional[List[str]] = None,
    max_cols: int = 2,
    size: float = 5.0,
    alpha: float = 0.9,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    use_per_panel_scale: bool = True,
    show_colorbar: bool = True,
    fig_width: float = 12.0,
    fig_height: Optional[float] = None,
) -> Tuple[plt.Figure, np.ndarray, np.ndarray]:
    """
    Plot KDE spatial density for features on embedding.
    
    Parameters
    ----------
    adata : AnnData
        Annotated data object
    features : list of str
        Feature names (transcript IDs, gene names, etc.)
    basis : str
        Embedding basis (e.g., 'umap', 'tsne', 'spatial')
    adjust : float
        Bandwidth adjustment for KDE
    cmaps : list of str, optional
        Colormap for each feature (default: 'Reds' for all)
    max_cols : int
        Maximum number of columns in grid
    size : float
        Point size
    alpha : float
        Point alpha
    vmin, vmax : float, optional
        Color scale limits (only used if use_per_panel_scale=False)
    use_per_panel_scale : bool
        If True (default), each panel has its own colorbar scale.
        If False, all panels share a global scale.
    show_colorbar : bool
        If True, add colorbar to each subplot
    fig_width : float
        Figure width in inches
    fig_height : float, optional
        Figure height in inches (auto if None)
    
    Returns
    -------
    fig : plt.Figure
        The figure object
    coords : np.ndarray
        Coordinates array (n_cells, 2)
    density_values : np.ndarray
        Density matrix (n_features, n_cells)
    """
    # Get coordinates
    coords = get_coords_from_adata(adata, basis=basis)
    
    # Calculate density for each feature
    density_values = []
    for feat in features:
        density = calculate_density(
            adata,
            feature=feat,
            basis=basis,
            adjust=adjust,
            map_to_cells=True,
        )
        density_values.append(density)
    
    density_values = np.array(density_values)  # (n_features, n_cells)
    
    # Default colormaps: Reds for density
    if cmaps is None:
        cmaps = ["Reds"] * len(features)
    
    # Strip version suffixes and add "(Density)" label
    titles = [_versionless(f) + " (Density)" for f in features]
    
    # Determine axis labels based on basis
    if basis.lower() == "umap":
        axis_labels = ("UMAP1", "UMAP2")
    elif basis.lower() == "tsne":
        axis_labels = ("tSNE1", "tSNE2")
    elif basis.lower() in ("spatial", "space"):
        axis_labels = ("X", "Y")
    else:
        axis_labels = (f"{basis}1", f"{basis}2")
    
    # Plot - density plots default to per-panel scale
    fig = plot_coords_standalone(
        coords,
        density_values,
        titles=titles,
        cmaps=cmaps,
        max_cols=max_cols,
        size=size,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax,
        use_per_panel_scale=use_per_panel_scale,
        axis_labels=axis_labels,
        show_colorbar=show_colorbar,
        fig_width=fig_width,
        fig_height=fig_height,
    )
    
    return fig, coords, density_values

## Scanpy-like API Examples
> Demonstration of plot_transcript_umap() and plot_umap_density()

## Spatial Plot Examples
> Demonstration of plot_transcript_spatial() and plot_spatial_density()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()